In [1]:
# pip install accelerate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

checkpoint = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# for fp16 use `torch_dtype=torch.float16` instead
model = AutoModelForCausalLM.from_pretrained(
    checkpoint, device_map="auto", torch_dtype=torch.bfloat16
)


/Users/tashi/Desktop/misc/learn-rlhf/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1479.80it/s]


In [2]:
inputs = tokenizer.encode("Start a sentence", return_tensors="pt", add_special_tokens=True).to("mps")
outputs = model.generate(
    inputs,
    max_new_tokens=32
)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Start a sentence with a capital letter.

The first sentence of a paragraph should be capitalized.

The second sentence of a paragraph should be capitalized.



In [3]:
from datasets import load_dataset

ds = load_dataset("HuggingFaceH4/no_robots", split="train")

In [4]:
ds[1]

{'prompt': 'Help write a letter of 100 -200 words to my future self for Kyra, reflecting on her goals and aspirations.',
 'prompt_id': '7d443ef2cc3e34d9dc6ffcdf748c1d2a9880cd48be9c9887df29d25be90123f4',
 'messages': [{'content': 'Help write a letter of 100 -200 words to my future self for Kyra, reflecting on her goals and aspirations.',
   'role': 'user'},
  {'content': "Dear Future Self,\n\nI hope you're happy and proud of what you've achieved. As I write this, I'm excited to think about our goals and how far you've come. One goal was to be a machine learning engineer. I hope you've worked hard and become skilled in this field. Keep learning and innovating. Traveling was important to us. I hope you've seen different places and enjoyed the beauty of our world. Remember the memories and lessons. Starting a family mattered to us. If you have kids, treasure every moment. Be patient, loving, and grateful for your family.\n\nTake care of yourself. Rest, reflect, and cherish the time you spe

In [5]:
print(tokenizer.chat_template)


None


The base model has no chat template, let's use the one from the official SmolInstruct model

In [6]:
chat_tmplt = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct").chat_template

In [19]:
tokenizer.chat_template=chat_tmplt

We pad tokens becuase examples in a batch have different lengths

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [44]:
tokenizer.pad_token

'<|endoftext|>'

Apply chat template to evertyhing before the final assistant answer and add the assistant prefix

In [45]:
prompt_ids = tokenizer.apply_chat_template(ds[1]["messages"][:-1], tokenize=True, add_generation_prompt=True)

In [46]:
prompt_ids

{'input_ids': [1, 9690, 198, 2683, 359, 253, 5356, 5646, 11173, 3365, 3511, 308, 34519, 28, 7018, 411, 407, 19712, 8182, 2, 198, 1, 4093, 198, 19523, 2965, 253, 4386, 282, 216, 33, 32, 32, 731, 34, 32, 32, 1924, 288, 957, 1774, 639, 327, 17365, 317, 28, 10329, 335, 874, 3949, 284, 17311, 30, 2, 198, 1, 520, 9531, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Apply chat template to the whole conversation, including the final assistant answer

In [47]:
full_ids = tokenizer.apply_chat_template(ds[1]["messages"], tokenize=True, add_generation_prompt=True)

In [48]:
full_ids

{'input_ids': [1, 9690, 198, 2683, 359, 253, 5356, 5646, 11173, 3365, 3511, 308, 34519, 28, 7018, 411, 407, 19712, 8182, 2, 198, 1, 4093, 198, 19523, 2965, 253, 4386, 282, 216, 33, 32, 32, 731, 34, 32, 32, 1924, 288, 957, 1774, 639, 327, 17365, 317, 28, 10329, 335, 874, 3949, 284, 17311, 30, 2, 198, 1, 520, 9531, 198, 35097, 9272, 9506, 28, 198, 198, 57, 3826, 346, 2316, 5587, 284, 9723, 282, 732, 346, 3543, 6551, 30, 1032, 339, 2965, 451, 28, 339, 5248, 8916, 288, 1510, 563, 653, 3949, 284, 638, 1869, 346, 3543, 1690, 30, 1963, 3491, 436, 288, 325, 253, 3746, 1380, 10776, 30, 339, 3826, 346, 3543, 4147, 1759, 284, 1438, 10632, 281, 451, 1955, 30, 5545, 1380, 284, 2992, 674, 30, 15179, 274, 436, 1070, 288, 468, 30, 339, 3826, 346, 3543, 2269, 896, 3373, 284, 9765, 260, 6122, 282, 653, 905, 30, 5258, 260, 8269, 284, 4431, 30, 21468, 253, 1564, 36682, 288, 468, 30, 1094, 346, 457, 3081, 28, 14226, 897, 3847, 30, 1934, 3158, 28, 15239, 28, 284, 18044, 327, 469, 1564, 30, 198, 198, 12653, 

In [105]:
from transformers import PreTrainedTokenizer


IGNORE_INDEX=-100

def encode_example(example, tokenizer: PreTrainedTokenizer, max_length):
    messages = example["messages"]

    # apply_chat_template(messages) != apply_chat_template(messages[-1:]) + apply_chat_template(messages[:-1])
    prompt_ids = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True,return_dict=False)

    full_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False,return_dict=False)


    labels = [IGNORE_INDEX] * len(prompt_ids)
    labels += full_ids[len(prompt_ids):]

    input_ids = full_ids[:max_length]
    labels = labels[:max_length]

    if all(x == IGNORE_INDEX for x in labels):
        return None

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }



In [106]:
print(encode_example(ds[1], tokenizer, 1200))

{'input_ids': tensor([    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
         3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
          198,     1,  4093,   198, 19523,  2965,   253,  4386,   282,   216,
           33,    32,    32,   731,    34,    32,    32,  1924,   288,   957,
         1774,   639,   327, 17365,   317,    28, 10329,   335,   874,  3949,
          284, 17311,    30,     2,   198,     1,   520,  9531,   198, 35097,
         9272,  9506,    28,   198,   198,    57,  3826,   346,  2316,  5587,
          284,  9723,   282,   732,   346,  3543,  6551,    30,  1032,   339,
         2965,   451,    28,   339,  5248,  8916,   288,  1510,   563,   653,
         3949,   284,   638,  1869,   346,  3543,  1690,    30,  1963,  3491,
          436,   288,   325,   253,  3746,  1380, 10776,    30,   339,  3826,
          346,  3543,  4147,  1759,   284,  1438, 10632,   281,   451,  1955,
           30,  5545,  1380,   284,  2992,   674, 

In [107]:
from torch.utils.data import Dataset

class SFTDataset(Dataset):
    def __init__(self, raw_dataset, tokenizer, max_length):
        self.examples = []

        for row in raw_dataset:
            encoded = encode_example(row, tokenizer, max_length)
            if encoded is not None:
                self.examples.append(encoded)

    
    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


In [108]:
test_ds = SFTDataset(ds, tokenizer, 1024)

In [119]:
torch.equal(test_ds[1]["labels"], encode_example(ds[1], tokenizer, 1024)["labels"])

True

In [ ]:
def collator(examples, pad_token_id):
    max_len = max(ex["input_ids"].size(0) for ex in examples)

    input_ids = []
    attention_mask = []
    labels= []

    for ex in examples:
        ids = ex["input_ids"]
        lbl = ex["labels"]
        pad_len = max_len - ids.size(0)

        input_ids.append(
            torch.cat([
                ids, 
                torch.full(
                    size=(pad_len, ),
                    fill_value=pad_token_id,
                    dtype=torch.long
                )
            ])
        )

        attention_mask.append(
            torch.cat([
                torch.ones(size=(ids.size(0), ), dtype=torch.long),
                torch.zeros(size=(pad_len, ), dtype=torch.long),
            ])
        )

        labels.append(
            torch.cat([
                lbl, 
                torch.full(size=(pad_len, ), fill_value=IGNORE_INDEX, dtype=torch.long)
            ])
        )

    return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attention_mask),
            "labels": torch.stack(labels),
        }

In [132]:
from torch.utils.data import DataLoader

train_dataset = SFTDataset(ds, tokenizer, max_length=2048)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collator
)